<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit2/session-08-loops-and-graphs/notebook.ipynb)


# Session 8 — Loops and graphs

**Goal:** build the same task three ways (chain, tool loop, reflection), then write the workflow as a state machine whose transitions are declared — so an edge nobody drew is refused instead of followed. *Threads: loop + graph engineering.*

Every cell runs offline on `FakeLLM`. Nothing here calls a provider or the network.


In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review

## 1. Way 1: the deterministic chain (no decisions)

Retrieve, prompt, parse. Every time, in that order. Nothing decides anything.

In [3]:
import json

from bootcamp_agent.documents import load_corpus
from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.retrieval import retrieve
from bootcamp_agent.schema import ANSWER_JSON_INSTRUCTIONS, parse_research_answer

documents = load_corpus(CORPUS_DIR)
QUESTION = "What stopping conditions should an agent loop have?"

ANSWER_JSON = json.dumps({
    "answer": "Stop on: final answer, empty retrieval, double parse failure, budget exhausted, repetition.",
    "citations": ["agent-loops"], "confidence": 0.9, "needs_human_review": False,
})
llm = FakeLLM(responses={"stopping": ANSWER_JSON})


def chain(question: str):
    scored = retrieve(question, documents, top_k=3)  # step 1, always
    context = "\n\n".join(f"[{s.chunk.doc_id}]\n{s.chunk.text}" for s in scored)
    raw = llm.complete(system=ANSWER_JSON_INSTRUCTIONS, user=f"{context}\n\nQ: {question}")
    return parse_research_answer(raw)  # step 3, always


print(chain(QUESTION).answer)
print(f"model calls so far: {len(llm.calls)}")

Stop on: final answer, empty retrieval, double parse failure, budget exhausted, repetition.
model calls so far: 1


## 2. Way 2: the tool loop (the model decides)

This is `answer_question`, the capstone's own loop. It can refuse before it spends a model call.

In [4]:
from bootcamp_agent.agent import answer_question

loop_llm = FakeLLM(responses={"stopping": ANSWER_JSON})
result = answer_question(QUESTION, documents, loop_llm)
print(result.answer.answer)
print([e.kind for e in result.trace], f"model calls: {len(loop_llm.calls)}")

Stop on: final answer, empty retrieval, double parse failure, budget exhausted, repetition.
['retrieve', 'llm_call', 'decision'] model calls: 1


## 3. Way 3: reflection (draft, critique, revise, once)

The critique is itself a model call. Every extra beat costs latency and money, so the comparison table decides whether it pays.

In [5]:
def reflect_once(question: str):
    calls_before = len(llm.calls)
    draft = chain(question)
    critique = llm.complete(
        system="You are a strict reviewer. Reply APPROVE or one concrete fix.",
        user=f"Q: {question}\nDraft: {draft.answer}",
    )
    if critique.strip().upper().startswith("APPROVE"):
        return draft, len(llm.calls) - calls_before
    revised = chain(question + f" (address: {critique[:80]})")
    return revised, len(llm.calls) - calls_before  # cap: exactly one revision, no loop


answer, calls = reflect_once(QUESTION)
print(f"calls used by reflection: {calls}")
print(answer.answer)

calls used by reflection: 3
Stop on: final answer, empty retrieval, double parse failure, budget exhausted, repetition.


## 4. Exercise: the comparison table

**Context.** Three ways, same task. Two of the columns are facts you can count from the cells above; one is judgement. The check judges the facts.

**Instructions.**

1. The `chain` row is done. Its call count came from `len(llm.calls)`, not from a guess.
2. Fill `tool_loop` and `reflection`: the call count, whether the pattern can refuse BEFORE a model call, and the failure modes it brings.
3. Run the check. It verifies the chain's count and all three refusal answers against how the code actually behaves.

In [6]:
comparison = {
    "chain": {
        "llm_calls": 1, "can_refuse_early": False,
        "failure_modes": "cannot adapt; bad retrieval flows straight into the answer",
    },
    "tool_loop": {
        "llm_calls": "1-2 (+retry)", 
        "can_refuse_early": True, 
        "failure_modes": "puede entrar en bucles de reintentos o generar argumentos invalidos",
    },
    "reflection": {
        "llm_calls": "2-3", 
        "can_refuse_early": True, 
        "failure_modes": "duplica la latencia y la critica puede alucinar correcciones erroneas",
    },
}


**Expected output** (yours may differ in wording, not in shape):

```
chain       calls=1            refuse_early=False
tool_loop   calls=1-2 (+retry) refuse_early=True
reflection  calls=2-3          refuse_early=True
✅ ch08-e1 passed
```

In [7]:
check("ch08-e1", comparison)

✅ ch08-e1 passed


True

## 5. Appendix: the same assistant as a graph

**Optional, and not part of the graded capstone.** Two scripts sit beside this notebook:

```bash
uv add langgraph   # not a course dependency

uv run python units/en/unit2/session-08-loops-and-graphs/langgraph_capstone.py   # the loop, as a graph
uv run python units/en/unit2/session-08-loops-and-graphs/langgraph_subagents.py  # four narrow roles
```

Both run on the `fake` lane at zero cost and exit cleanly with a printed reason if langgraph is absent. What a graph buys is a declared state and declared edges: "what happens after a parse failure" becomes a line you point at. What it costs is a dependency and a call count that grows with every node.

Read the traces, then decide. The subagents script is the sharper lesson: its router stops an out-of-corpus question after ONE call, where the four-role path spends three.

## 6. Exercise: the graph comparison

**Context.** The appendix is only worth anything if you compare it. Either you ran it and can name the difference, or you did not and can say why. Both are honest; a blank is not.

**Instructions.**

1. Run at least one of the two scripts, or decide not to.
2. If it ran: set `ran` to True, and record the model-call count for the framework-free loop (from clinic 3's trace or chapter 8) and for the graph (the script prints it).
3. Write the difference you actually observed in the traces.
4. If it did not run: leave `ran` False and write why in `skipped_because`. The check accepts that, and refuses a blank.

In [8]:
graph_comparison = {
    "ran": False,  # TODO(you): True if you ran a script
    "skipped_because": "langgraph no esta instalado y se prefiere mantener la implementacion libre de dependencias externas",  # TODO(you): if you did not run it, why
    "framework_free_calls": 0,  # TODO(you): calls the plain loop spent
    "graph_calls": 0,  # TODO(you): calls the graph spent
    "difference": "",  # TODO(you): what the traces showed
}
print(graph_comparison)

{'ran': False, 'skipped_because': 'langgraph no esta instalado y se prefiere mantener la implementacion libre de dependencias externas', 'framework_free_calls': 0, 'graph_calls': 0, 'difference': ''}


**Expected output** (yours may differ in wording, not in shape):

```
{'ran': True, 'framework_free_calls': 1, 'graph_calls': 3, ...}
✅ ch08-e2 passed
```

In [9]:
check("ch08-e2", graph_comparison)

✅ ch08-e2 passed


True

## 7. The graph, written down

The appendix used a framework to declare states and edges. You do not need one. Five states and six edges are a dict, and a dict is something you can read, diff and test.

```mermaid
graph LR
    P[planning] -->|plan_ready| R[retrieving]
    P -->|out_of_scope| F[refusing]
    R -->|hits| A[answering]
    R -->|no_hits| F
    A -->|answered| D[done]
    F -->|refused| D
```

Two things are already visible in the drawing. There is no edge from `refusing` to `answering`, so a refusal can never turn into an answer. And `done` has no outgoing arrow at all, which is the whole of "terminal".

In [10]:
STATES = ("planning", "retrieving", "answering", "refusing", "done")

# The graph as data. A pair that is not a key here is not an edge, and there is
# no other place an edge could be hiding.
TRANSITIONS = {
    ("planning", "plan_ready"): "retrieving",
    ("planning", "out_of_scope"): "refusing",
    ("retrieving", "hits"): "answering",
    ("retrieving", "no_hits"): "refusing",
    ("answering", "answered"): "done",
    ("refusing", "refused"): "done",
}
START = {"state": "planning", "visited": ["planning"], "rejected": None}

for (source, event), target in TRANSITIONS.items():
    print(f"{source:11} --{event:13}--> {target}")
print(f"\nstates with no outgoing edge: {[s for s in STATES if not any(s == a for a, _ in TRANSITIONS)]}")

planning    --plan_ready   --> retrieving
planning    --out_of_scope --> refusing
retrieving  --hits         --> answering
retrieving  --no_hits      --> refusing
answering   --answered     --> done
refusing    --refused      --> done

states with no outgoing edge: ['done']


## 8. Exercise: typed transitions

**Context.** `step(state, event)` takes one state and one event and returns the next state. It reads the table and nothing else. The state is a dict:

```python
{"state": "retrieving", "visited": ["planning", "retrieving"], "rejected": None}
```

| Key | What it holds |
|---|---|
| `state` | one of the five names, never anything else |
| `visited` | the states entered, in order, starting with `planning` — it only ever grows |
| `rejected` | `None` after a legal move; a sentence after a refused one |

An event whose `(state, name)` pair is missing from the table is refused: the state comes back unchanged, `rejected` says which event was refused and from where, and nothing is raised. `done` has no row in the table, so the same branch is what makes it terminal.

**Instructions.**

1. The lookup is done. Fill TODO 1: on a missing pair, return `state` and `visited` exactly as they came in, and write `rejected` as a sentence naming the event and the state. A caller reads this; "invalid transition" tells them nothing.
2. Fill TODO 2: on a legal move, append `target` to `visited`. Append — never rebuild the list from the current state, or the run's history disappears and with it every cap you could put on a retry.
3. Run the check. It drives your `step` over all five states and every event, legal and not: the answer path, both refusal paths, every event fired at `done`, and every pair the table never declared.

In [11]:
def step(state: dict, event: dict) -> dict:
    """One transition. The table decides; an undeclared pair is refused."""
    current, name = state["state"], event["name"]
    target = TRANSITIONS.get((current, name))
    if target is None:
        # TODO(you) 1: nothing moves. Return `current` and the SAME visited list,
        #              with a `rejected` sentence naming `name` and `current`.
        return {
            "state": current,
            "visited": list(state["visited"]),
            "rejected": f"'{name}' is not a declared event in state '{current}'",
        }
    # TODO(you) 2: you entered `target` — append it to visited.
    return {
        "state": target,
        "visited": [*state["visited"], target],
        "rejected": None,
    }


planned = step(START, {"name": "plan_ready"})
print(f"plan_ready  -> {planned['state']!r}  visited={planned['visited']}")
refused = step(START, {"name": "answered"})
print(f"answered    -> {refused['state']!r}  rejected={refused['rejected']!r}")


plan_ready  -> 'retrieving'  visited=['planning', 'retrieving']
answered    -> 'planning'  rejected="'answered' is not a declared event in state 'planning'"


**Expected output** (yours may differ in wording, not in shape):

```
plan_ready  -> 'retrieving'  visited=['planning', 'retrieving']
answered    -> 'planning'  rejected="'answered' is not a declared event in state 'planning'"
✅ ch08-e3 passed
```

In [12]:
check("ch08-e3", step)

✅ ch08-e3 passed


True

## 9. The run, read back

An event log driven through `step`. Two of these events are illegal, and the interesting part is what the machine does with them: it stays put and says so, and the run carries on from where it was.

In [13]:
EVENT_LOG = [
    {"name": "plan_ready"},
    {"name": "answered"},  # illegal: nothing has been retrieved yet
    {"name": "hits"},
    {"name": "answered"},
    {"name": "refused"},  # illegal: the run is done
]

state = dict(START)
for event in EVENT_LOG:
    state = step(state, event)
    print(f"{event['name']:12} -> state={str(state['state']):12} rejected={state.get('rejected')}")
print(f"\nvisited: {' -> '.join(str(entry) for entry in state['visited'])}")

plan_ready   -> state=retrieving   rejected=None
answered     -> state=retrieving   rejected='answered' is not a declared event in state 'retrieving'
hits         -> state=answering    rejected=None
answered     -> state=done         rejected=None
refused      -> state=done         rejected='refused' is not a declared event in state 'done'

visited: planning -> retrieving -> answering -> done


## 10. Failure injection: a retry storm

Add one edge — `retrieving --retry--> retrieving` — and the graph is still perfectly legal, and it never ends. A retry storm is not a bug in the retry; it is an edge with no counter on it.

The counter is already in your hand. `visited` only grows, so the number of times a state was entered is a fact about the run, and a cap is a rule about that fact.

In [14]:
STORM = {**TRANSITIONS, ("retrieving", "retry"): "retrieving"}


def storm_step(state: dict, event: dict, table: dict) -> dict:
    """`step`, with the table passed in, so one extra edge is one line to change."""
    target = table.get((state["state"], event["name"]))
    if target is None:
        return {**state, "rejected": f"{event['name']!r} is not declared in {state['state']!r}"}
    return {"state": target, "visited": [*state["visited"], target], "rejected": None}


state = {"state": "retrieving", "visited": ["planning", "retrieving"], "rejected": None}
for _ in range(8):
    state = storm_step(state, {"name": "retry"}, STORM)
print(f"unbounded: entered 'retrieving' {state['visited'].count('retrieving')} times, still there")

MAX_VISITS = 3  # the first pass, plus two retries
state = {"state": "retrieving", "visited": ["planning", "retrieving"], "rejected": None}
for _ in range(8):
    if state["visited"].count("retrieving") >= MAX_VISITS:
        state = {**state, "rejected": f"'retrieving' entered {MAX_VISITS} times; retries spent"}
        break
    state = storm_step(state, {"name": "retry"}, STORM)
print(f"capped:    visited={state['visited']}")
print(f"           rejected={state['rejected']!r}")

unbounded: entered 'retrieving' 9 times, still there
capped:    visited=['planning', 'retrieving', 'retrieving', 'retrieving']
           rejected="'retrieving' entered 3 times; retries spent"


## 11. Failure injection: a partial tool failure

Two retrieval sources, one of them down. Half the passages came back. The question the graph forces you to answer is which event this is — `hits` or `no_hits` — and that is a decision, not an accident. Make it in the open, record what failed, and let the state machine route it.

In [15]:
from bootcamp_agent.tools import ToolError


def corpus_source(question: str) -> list[str]:
    hits = [s.chunk.doc_id for s in retrieve(question, documents, top_k=2)]
    return list(dict.fromkeys(hits))  # one entry per document, in rank order


def wiki_source(question: str) -> list[str]:
    raise ToolError("wiki: the index went away mid-run")


def gather(question: str) -> tuple[list[str], list[str]]:
    """Every source that answers, plus every source that did not."""
    found, failed = [], []
    for name, source in (("corpus", corpus_source), ("wiki", wiki_source)):
        try:
            found.extend(source(question))
        except ToolError as error:
            failed.append(f"{name} ({error})")
    return found, failed


found, failed = gather(QUESTION)
event = {"name": "hits" if found else "no_hits", "partial": bool(failed)}
after = step({"state": "retrieving", "visited": ["planning", "retrieving"], "rejected": None}, event)
print(f"found={found}  failed={failed}")
print(f"event={event['name']!r} partial={event['partial']}  ->  state={after['state']!r}")
print("the answer that follows is drafted from half a corpus; the flag has to travel with it")

found=['agent-loops']  failed=['wiki (wiki: the index went away mid-run)']
event='hits' partial=True  ->  state='answering'
the answer that follows is drafted from half a corpus; the flag has to travel with it


## Exit ticket

One thing that works, one thing that is unclear, your next action.

Homework: draw the state machine of something you have shipped, including the failure edges, and mark every edge nobody declared. Then read `docs/guides/graph-engineering.md`.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [16]:
review("ch08")

ch08: 3/3 passed  ·  300/300 marks


True